In [1]:
print("Hello WOrld")

Hello WOrld


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd()
for candidate in (cwd, cwd.parent):
    if (candidate / "helpers" / "common.py").exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))
        break

## For this one, i set the temp=0
from helpers.common import together_ai_llm, langfuse, langfuse_handler

/home/yash/Desktop/Code/1. Langchain and Ollama/lang_ollama_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Setup the DB

In [3]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///employees_db-full-1.0.6.db")
db.dialect
db.get_usable_table_names()

try:
    tables = db.get_usable_table_names()
    print("✓ Database connected successfully")
    print(f"✓ Found {len(tables)} tables: {', '.join(tables)}")
    
except Exception as e:
    print(f"✗ Database connection failed: {e}")

db.run("Select count(*) from employees")
SCHEMA = db.get_table_info()

✓ Database connected successfully
✓ Found 6 tables: departments, dept_emp, dept_manager, employees, salaries, titles


#### Setup Tools

In [4]:
from langchain.tools import tool

@tool
def get_database_schema(table_name: str = None) -> str:
    """Get database schema information for SQL query generation.
    Use this first to understand table structure before creating queries."""
    print(f"Getting schema for: {table_name if table_name else 'all tables'}")
    
    if table_name:
        try:
            tables = db.get_usable_table_names()
            if table_name.lower() in [t.lower() for t in tables]:
                result = db.get_table_info([table_name])
                print(f"✓ Retrieved schema for table: {table_name}")
                return result
            else:
                return f"Error: Table '{table_name}' not found. Available tables: {', '.join(tables)}"
        except Exception as e:
            return f"Error getting table info: {e}"
    else:
        print("✓ Retrieved full database schema")
        return SCHEMA
get_database_schema.invoke({'table_name': "dept_emp"})
get_database_schema.invoke({})

Getting schema for: dept_emp
✓ Retrieved schema for table: dept_emp
Getting schema for: all tables
✓ Retrieved full database schema


'\nCREATE TABLE departments (\n\tdept_no CHAR(4) NOT NULL, \n\tdept_name VARCHAR(40) NOT NULL, \n\tPRIMARY KEY (dept_no), \n\tUNIQUE (dept_name)\n)\n\n/*\n3 rows from departments table:\ndept_no\tdept_name\nd009\tCustomer Service\nd005\tDevelopment\nd002\tFinance\n*/\n\n\nCREATE TABLE dept_emp (\n\temp_no INTEGER NOT NULL, \n\tdept_no CHAR(4) NOT NULL, \n\tfrom_date DATE NOT NULL, \n\tto_date DATE NOT NULL, \n\tPRIMARY KEY (emp_no, dept_no), \n\tFOREIGN KEY(dept_no) REFERENCES departments (dept_no), \n\tFOREIGN KEY(emp_no) REFERENCES employees (emp_no)\n)\n\n/*\n3 rows from dept_emp table:\nemp_no\tdept_no\tfrom_date\tto_date\n10001\td005\t1986-06-26\t9999-01-01\n10002\td007\t1996-08-03\t9999-01-01\n10003\td004\t1995-12-03\t9999-01-01\n*/\n\n\nCREATE TABLE dept_manager (\n\tdept_no CHAR(4) NOT NULL, \n\temp_no INTEGER NOT NULL, \n\tfrom_date DATE NOT NULL, \n\tto_date DATE NOT NULL, \n\tPRIMARY KEY (emp_no, dept_no), \n\tFOREIGN KEY(dept_no) REFERENCES departments (dept_no), \n\tFOREIG

In [ ]:

@tool
def generate_sql_query(question: str, schema_info: str = None) -> str:
    """Generate a SQL SELECT query from a natural language question using database schema.
    Always use this after getting schema information."""
    print(f"Generating SQL for: {question[:100]}...")
    
    schema_to_use = schema_info if schema_info else SCHEMA
    
    prompt = f"""
                Based on this database schema:
                {schema_to_use}

                Generate a SQL query to answer this question: {question}

                Rules:
                - Use only SELECT statements
                - Include only existing columns and tables
                - Add appropriate WHERE, GROUP BY, ORDER BY clauses as needed
                - Limit results to 10 rows unless specified otherwise
                - Use proper SQL syntax for SQLite

                Return only the SQL query, nothing else.
                """
    
    try:
        response = together_ai_llm.invoke(prompt)
        query = response.content.strip()
        print("Generated SQL query")
        return query
    except Exception as e:
        return f"Error generating query: {e}"

result = generate_sql_query.invoke({"question": "what is maximum salary in employees"})
print(result)

🔧 Generating SQL for: what is maximum salary in employees...
Generated SQL query
SELECT MAX(salary) AS max_salary
FROM salaries;


In [7]:
db.run(result)

'[(158220,)]'

In [19]:
import re
import sqlparse

@tool
def validate_sql_query(query: str) -> str:
    """Validate SQL query for safety before execution."""

    try:
        print(f"Validating SQL: {query[:100]}...")

        clean_query = query.strip()

        clean_query = re.sub(r"```sql\s*", "", clean_query, flags=re.IGNORECASE)
        clean_query = re.sub(r"```", "", clean_query)

        clean_query = re.sub(r"--.*", "", clean_query)
        clean_query = re.sub(r"/\*.*?\*/", "", clean_query, flags=re.DOTALL)

        clean_query = clean_query.strip().rstrip(";")

        if ";" in clean_query:
            return "Error: Multiple statements are not allowed"

        parsed = sqlparse.parse(clean_query)

        if not parsed:
            return "Error: Invalid SQL"

        stmt = parsed[0]

        if stmt.get_type() != "SELECT":
            return "Error: Only SELECT statements are allowed"

        query_upper = clean_query.upper()

        match = re.search(r"LIMIT\s+(\d+)", query_upper)
        if match and int(match.group(1)) > 1000:
            return "Error: LIMIT exceeds maximum allowed (1000)"

        print("Query validation passed")
        return f"Valid: {clean_query}"

    except Exception as e:
        return f"Error: {str(e)}"
validate_sql_query.invoke({"query": "SELECT MAX(salary) AS max_salary"})
# validate_sql_query.invoke({"query": "DELETE MAX(salary) AS max_salary"})

Validating SQL: SELECT MAX(salary) AS max_salary...
Query validation passed


'Valid: SELECT MAX(salary) AS max_salary'

In [21]:

@tool
def execute_sql_query(query: str) -> str:
    """Execute a validated SQL query and return results.
    Only use this after validating the query for safety."""
    print(f"Executing SQL: {query[:100]}...")
    
    try:
        clean_query = query.strip()
        if clean_query.startswith("Valid: "):
            clean_query = clean_query[7:]  # Remove "Valid: " prefix
        
        clean_query = re.sub(r'```sql\s*', '', clean_query, flags=re.IGNORECASE)
        clean_query = re.sub(r'```\s*', '', clean_query)
        clean_query = clean_query.strip().rstrip(";")
        
        result = db.run(clean_query)
        print("Query executed successfully")
        
        if result:
            return f"Query Results:\n{result}"
        else:
            return "Query executed successfully but returned no results."
            
    except Exception as e:
        error_msg = f"Execution Error: {str(e)}"
        print(f"✗ {error_msg}")
        return error_msg
execute_sql_query.invoke({"query": "SELECT MAX(salary) FROM salaries;"})

Executing SQL: SELECT MAX(salary) FROM salaries;...
Query executed successfully


'Query Results:\n[(158220,)]'

In [22]:
@tool
def fix_sql_error(original_query: str, error_message: str, question: str) -> str:
    """Fix a failed SQL query by analyzing the error and generating a corrected version.
    Use this when validation or execution fails."""
    print(f"Fixing SQL error: {error_message[:100]}...")
    
    fix_prompt = f"""
                    The following SQL query failed:
                    Query: {original_query}
                    Error: {error_message}
                    Original Question: {question}

                    Database Schema:
                    {SCHEMA}

                    Analyze the error and provide a corrected SQL query that:
                    1. Fixes the specific error mentioned
                    2. Still answers the original question
                    3. Uses only valid table and column names from the schema
                    4. Follows SQLite syntax rules

                    Return only the corrected SQL query, nothing else.
                    """
    
    try:
        response = together_ai_llm.invoke(fix_prompt)
        fixed_query = response.content.strip()
        print("✓ Generated fixed SQL query")
        return fixed_query
    except Exception as e:
        return f"Error generating fix: {e}"

fix_sql_error.invoke({"original_query": "Update MAX(salary) FROM salaries;", "error_message": "Error: Only SELECT statements are allowed", "question": "what is maximum salary in employees"})

Fixing SQL error: Error: Only SELECT statements are allowed...
✓ Generated fixed SQL query


'SELECT MAX(salary) AS max_salary FROM salaries;'